Import Libraries

In [1]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from flax import linen as nn
from evojax.util import get_params_format_fn

import time
import numpy as np
import pandas as pd
from scipy import io
import matplotlib.pyplot as plt

# choose GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
#jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

Problem: Navier-Stokes Equation

        u*u_x + v*u_y - 1/Re*(u_xx+u_yy) + p_x = 0
        v*v_x + v*v_y - 1/Re*(v_xx+v_yy) + p_y = 0
        u_x + v_y = 0

In [2]:
# parameter
Re = 5000

In [3]:
mat_data = io.loadmat('ldc_Re5000.mat')
print("x: ", mat_data['x'].shape, ", y: " , mat_data['y'].shape,", u: ", mat_data['u'].shape, ", v: " , mat_data['v'].shape)

x:  (1, 256) , y:  (1, 256) , u:  (256, 256) , v:  (256, 256)


In [4]:
xx = mat_data['x'].reshape(-1,1)
yy = mat_data['y'].reshape(-1,1)
x_256, y_256 = jnp.meshgrid(xx[:,0],yy[:,0],indexing='ij')
u_256 = mat_data['u']
v_256 = mat_data['v']

2025-09-09 11:38:40.989623: W external/xla/xla/service/gpu/nvptx_compiler.cc:763] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.6.77). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [5]:

sim_x = x_256.reshape(-1,1)
sim_y = y_256.reshape(-1,1)

sim_u = u_256.reshape(-1,1)
sim_v = v_256.reshape(-1,1)
x_l, x_u, y_l, y_u = np.min(x_256), np.max(x_256), np.min(y_256), np.max(y_256)

ext = [x_l, x_u, y_l, y_u]
print (x_l, x_u, y_l, y_u)

data_X, data_Y = np.hstack([sim_x, sim_y]), np.hstack([sim_u, sim_v])

dx = dy = xx[2]-xx[1]
print (dx, dy)

# exclude corner points
corners = (sim_x == x_l) & (sim_y == y_u) | (sim_x == x_u) & (sim_y == y_u) | (sim_x == x_l) & (sim_y == y_l) | (sim_x == x_u) & (sim_y == y_l)


# pressure ref
x_ref, y_ref = np.unique(sim_x), np.unique(sim_y)
x_ref, y_ref = np.take(x_ref, x_ref.size //2), np.take(y_ref, y_ref.size //2)
print (x_ref, y_ref)

# split into BC data
bc = (data_X[:,0] == x_l) | (data_X[:,0] == x_u) | (data_X[:,1] == y_l) | (data_X[:,1] == y_u)
data_X_BC, data_Y_BC = data_X[bc], data_Y[bc]
print (data_X.shape, data_Y.shape, data_X_BC.shape, data_Y_BC.shape)

0.0 1.0 0.0 1.0
[0.00392157] [0.00392157]
0.5019608 0.5019608
(65536, 2) (65536, 2) (1020, 2) (1020, 2)


In [6]:
# convert to jnp
data_X, data_Y, data_X_BC, data_Y_BC = jnp.array(data_X), jnp.array(data_Y), jnp.array(data_X_BC), jnp.array(data_Y_BC)
print (data_X.shape, data_Y.shape, data_X_BC.shape, data_Y_BC.shape)

(65536, 2) (65536, 2) (1020, 2) (1020, 2)


DNN / PINN   

In [7]:
nn_acf = nn.silu

class PINN(nn.Module):
    """PINNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                    nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]     

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]
        def get_uvp(x, y):
            inputs = jnp.hstack([x,y-1])
            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(2*jnp.pi*hidden)
            # hidden = jnp.sin(hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p)
            return (u, v, p)  
    
        u, v, p = get_uvp(x, y)


        # axillary PDE outputs
        # discretization (1st order)
        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps = get_uvp(x, ys)

        uEbc, vEbc = 0, 0
        uWbc, vWbc = 0, 0
        uNbc, vNbc = 1, 0
        uSbc, vSbc = 0, 0

        
        if(bi_count == 1):
         
            xB_E, xB_W = abs(xE-x_u)<0.01*dx, abs(xW-x_l)<0.01*dx
            yB_N, yB_S = abs(yN-y_u)<0.01*dy, abs(yS-y_l)<0.01*dy           
            # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
            uE, vE = jnp.where(xB_E, uEbc, uE), jnp.where(xB_E, vEbc, vE)
            uW, vW = jnp.where(xB_W, uWbc, uW), jnp.where(xB_W, vWbc, vW)
            uN, vN = jnp.where(yB_N, uNbc, uN), jnp.where(yB_N, vNbc, vN)
            uS, vS = jnp.where(yB_S, uSbc, uS), jnp.where(yB_S, vSbc, vS)

        if(bi_count == 1):
         
            xB_e, xB_w = abs(xe-x_u)<0.01*dx, abs(xw-x_l)<0.01*dx
            yB_n, yB_s = abs(yn-y_u)<0.01*dy, abs(ys-y_l)<0.01*dy           
            # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
            ue, ve = jnp.where(xB_e, uEbc, ue), jnp.where(xB_e, vEbc, ve)
            uw, vw = jnp.where(xB_w, uWbc, uw), jnp.where(xB_w, vWbc, vw)
            un, vn = jnp.where(yB_n, uNbc, un), jnp.where(yB_n, vNbc, vn)
            us, vs = jnp.where(yB_s, uSbc, us), jnp.where(yB_s, vSbc, vs)

        ae = ue
        aw = -uw
        an = vn
        aas = -vs
        
        source_x = (pe - pw)
        source_y = (pn - ps) 

        bc  = (x == x_l) | (x == x_u) | (y == y_l) | (y == y_u)
        nbc = (~bc)
        
        div = (uE - uW + vN - vS)/2


        mom_x = ae*ue + aw*uw + an*un + aas*us + source_x - (uE + uW + uN + uS - 4*u)/(dx*Re) 
        mom_y = ae*ve + aw*vw + an*vn + aas*vs + source_y - (vE + vW + vN + vS - 4*v)/(dx*Re) 



        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx 
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx
        
        residuals_continuity = div/dx
        residuals_momentum_1 = mom_x/dx
        residuals_momentum_2 = mom_y/dy 

        
        outputs = jnp.hstack([u, v, p, residuals_continuity, residuals_momentum_1, residuals_momentum_2, bc, nbc,res_u,res_v])
        return outputs        
    
class DNN(nn.Module):
    """DNNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]    

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]
        def get_uvp(x, y):
            inputs = jnp.hstack([x,y-1])
            # feature mapping
            hidden = self.feats(inputs)
            # hidden = jnp.sin(hidden)
            hidden = jnp.sin(2*jnp.pi*hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p)
            return (u, v, p)  
 
        u, v, p = get_uvp(x, y)

       # axillary PDE outputs
        # discretization (1st order)
        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps = get_uvp(x, ys)

        uEbc, vEbc = 0, 0
        uWbc, vWbc = 0, 0
        uNbc, vNbc = 1, 0
        uSbc, vSbc = 0, 0
    
        if(bi_count == 1):
         
            xB_E, xB_W = abs(xE-x_u)<0.01*dx, abs(xW-x_l)<0.01*dx
            yB_N, yB_S = abs(yN-y_u)<0.01*dy, abs(yS-y_l)<0.01*dy           
            # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
            uE, vE = jnp.where(xB_E, uEbc, uE), jnp.where(xB_E, vEbc, vE)
            uW, vW = jnp.where(xB_W, uWbc, uW), jnp.where(xB_W, vWbc, vW)
            uN, vN = jnp.where(yB_N, uNbc, uN), jnp.where(yB_N, vNbc, vN)
            uS, vS = jnp.where(yB_S, uSbc, uS), jnp.where(yB_S, vSbc, vS)

        if(bi_count == 1):
         
            xB_e, xB_w = abs(xe-x_u)<0.01*dx, abs(xw-x_l)<0.01*dx
            yB_n, yB_s = abs(yn-y_u)<0.01*dy, abs(ys-y_l)<0.01*dy           
            # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
            ue, ve = jnp.where(xB_e, uEbc, ue), jnp.where(xB_e, vEbc, ve)
            uw, vw = jnp.where(xB_w, uWbc, uw), jnp.where(xB_w, vWbc, vw)
            un, vn = jnp.where(yB_n, uNbc, un), jnp.where(yB_n, vNbc, vn)
            us, vs = jnp.where(yB_s, uSbc, us), jnp.where(yB_s, vSbc, vs)

        # if 




        ae = ue
        aw = -uw
        an = vn
        aas = -vs
        
        source_x = (pe - pw)
        source_y = (pn - ps) 

        bc  = (x == x_l) | (x == x_u) | (y == y_l) | (y == y_u)
        nbc = (~bc)
        
        div = (uE - uW + vN - vS)/2

        mom_x = ae*ue + aw*uw + an*un + aas*us + source_x - (uE + uW + uN + uS - 4*u)/(dx*Re) 
        mom_y = ae*ve + aw*vw + an*vn + aas*vs + source_y - (vE + vW + vN + vS - 4*v)/(dx*Re) 
        
        residuals_continuity = div/dx
        residuals_momentum_1 = mom_x/dx
        residuals_momentum_2 = mom_y/dy 
        

        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx #-u*div
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx

        #shift p_ref
        x_pref, y_pref = jnp.ones_like(x)*x_ref,jnp.ones_like(y)*y_ref
        _, _, pref = get_uvp(x_pref,y_pref)

        pout = p - pref
        
        outputs = jnp.hstack([u, v, pout,residuals_momentum_1,residuals_momentum_2,res_u,res_v ])
        return outputs    


In [8]:
# choose seed
seed = 10
key, rng = random.split(random.PRNGKey(seed))

# dummy input
a = random.normal(key, [1,2])

# initialization call
bi_count = 1
n_nodes = 32
model, model_0 = PINN(), DNN()
params = model.init(key, a) 
num_params, format_params_fn = get_params_format_fn(params)
print (num_params)

# flatten initial params
params = jax.flatten_util.ravel_pytree(params)[0]  

params_0 = params 

12928


In [9]:
# minibatch (set #sample)
BS_ALL = int(len(data_X)*0.1)
BS_BC = int(len(data_X_BC)*0.1)

BS_PDE = BS_ALL # - BS_BC

n_all, n_bc = len(data_X), len(data_X_BC)
print(BS_ALL, BS_BC)
@jit
def minibatch(key):
    batch_all = random.choice(key, n_all, (BS_PDE,),replace=False)
    batch_bc = random.choice(key, n_bc, (BS_BC,),replace=False)   
    batch_X = jnp.vstack([data_X[batch_all], data_X_BC[batch_bc]])
    batch_Y = jnp.vstack([data_Y[batch_all], data_Y_BC[batch_bc]])
    
    return (batch_X, batch_Y)
    

# a,b,c = minibatch(key)

6553 102


In [10]:
# loss function
def eval_loss(params,params_0, inputs, labels):
    pred = model.apply(format_params_fn(params), inputs)
    u, v, p, residuals_continuity, residuals_momentum_1, residuals_momentum_2, bc, nbc,res_u,res_v = jnp.split(pred, 10, axis=1)
    gt_u, gt_v = jnp.split(labels, 2, axis=1)
    # stable evolution
    # PDE
    pred0 = model_0.apply(format_params_fn(params_0), inputs)
    u_0, v_0, p_0,residuals_momentum_10,residuals_momentum_20,res_u0,res_v0 = jnp.split(pred0, 7, axis=1) 
    beta_u = 0.79
    beta_v = 0.79

    loss_u = u - (beta_u*(-residuals_momentum_1 + res_u0 - res_u)*dx*dx*Re/4) - u_0
    loss_v = v - (beta_v*(-residuals_momentum_2 + res_v0 - res_v)*dx*dx*Re/4) - v_0


    alpha_u = 5.0
    alpha_v = 5.0

    pde_uvp  = jnp.square(residuals_continuity) + jnp.square(residuals_momentum_1) + jnp.square(residuals_momentum_2) +  alpha_u*jnp.abs(loss_u) + alpha_v*jnp.abs(loss_v)  
    pde_loss = jnp.sum(pde_uvp*nbc) / nbc.sum()
    # BC
    bc_u = (u - gt_u)
    bc_v = (v - gt_v)
    bc_uv   = jnp.square(bc_u) + jnp.square(bc_v)
    bc_loss = jnp.sum(bc_uv*bc) / bc.sum()
    # print(bc.sum())
    # mse
    uv = jnp.hstack([u, v])
    gt_uv = jnp.hstack([gt_u, gt_v])
    mse = jnp.mean(jnp.square(uv - gt_uv))
    rl2 = jnp.linalg.norm(uv - gt_uv) / jnp.linalg.norm(gt_uv)
    pre_V = jnp.sqrt(jnp.square(u) + jnp.square(v))
    gt_V = jnp.sqrt(jnp.square(gt_u) + jnp.square(gt_v))
    RL2 = jnp.linalg.norm(pre_V - gt_V) / jnp.linalg.norm(gt_V)
    loss = pde_loss*1  + 1*bc_loss
    return loss, (mse, RL2,pde_loss, bc_loss)

loss_grad = jax.jit(jax.value_and_grad(eval_loss, has_aux=True))    

In [11]:
# weights update  
@jit
def update(params, params_0,opt_state, key):
    batch_X, batch_Y = minibatch(key)
    (loss, (mse, rl2,pde_loss,bc_loss)), grad = loss_grad(params,params_0, batch_X, batch_Y)
    updates, opt_state = optimizer.update(grad, opt_state)
    params_0 = params # update u_0

    params = optax.apply_updates(params, updates)
    return params, params_0,opt_state, loss, mse, rl2, pde_loss, bc_loss

In [12]:
# optimizer
max_iters = 50000
max_lr = 1e-3
lr_scheduler = optax.warmup_cosine_decay_schedule(init_value=max_lr, peak_value=max_lr, warmup_steps=0,  
                                                  decay_steps=max_iters, end_value=1e-10,exponent=1.0)
optimizer = optax.adam(learning_rate=lr_scheduler) # Choose the method
opt_state = optimizer.init(params)

Training

In [13]:
runtime = 0
train_iters = 0

store = []
# max_iters = 4000
while (train_iters <= max_iters):
    # mini-batch update
    start = time.time()
    key, rng = random.split(rng) # update random generator
    params,params_0, opt_state, loss, mse, rl2 ,pde_loss,bc_loss= update(params,params_0, opt_state, key)
    end = time.time()
    runtime += (end-start)    
    # append weights
    if (train_iters % 2000 == 0):
        print ('iter. = %05d,  time = %03ds,  loss = %.2e  |  mse = %.2e,  rl2 = %.2e  , pde =  %.2e, bc = %.2e '%(train_iters, runtime, loss, mse, rl2,pde_loss,bc_loss))
        store.append([train_iters, runtime, loss, mse, rl2])
    train_iters += 1

store = jnp.array(store)

iter. = 00000,  time = 009s,  loss = 3.29e+02  |  mse = 4.83e-01,  rl2 = 2.52e+00  , pde =  3.28e+02, bc = 6.95e-01 
iter. = 02000,  time = 018s,  loss = 8.21e-02  |  mse = 4.03e-02,  rl2 = 7.69e-01  , pde =  5.99e-02, bc = 2.22e-02 
iter. = 04000,  time = 026s,  loss = 3.99e-02  |  mse = 4.45e-02,  rl2 = 8.05e-01  , pde =  3.28e-02, bc = 7.07e-03 
iter. = 06000,  time = 034s,  loss = 2.86e-02  |  mse = 3.96e-02,  rl2 = 8.19e-01  , pde =  2.30e-02, bc = 5.66e-03 
iter. = 08000,  time = 043s,  loss = 2.03e-02  |  mse = 3.27e-02,  rl2 = 7.44e-01  , pde =  1.52e-02, bc = 5.03e-03 
iter. = 10000,  time = 051s,  loss = 2.36e-02  |  mse = 1.09e-02,  rl2 = 4.69e-01  , pde =  2.01e-02, bc = 3.58e-03 
iter. = 12000,  time = 060s,  loss = 1.48e-02  |  mse = 2.04e-03,  rl2 = 2.09e-01  , pde =  1.17e-02, bc = 3.09e-03 
iter. = 14000,  time = 068s,  loss = 1.03e-02  |  mse = 6.05e-04,  rl2 = 1.11e-01  , pde =  8.05e-03, bc = 2.24e-03 
iter. = 16000,  time = 077s,  loss = 8.55e-03  |  mse = 6.27e-04

PINN solution

In [14]:
inputs, labels = data_X, data_Y

# print(inputs.shape)
uvp = model.apply(format_params_fn(params), inputs)
u, v, p = uvp[:,0:1], uvp[:,1:2], uvp[:,2:3]
gt_u, gt_v = jnp.split(labels, 2, axis=-1)


uv = jnp.hstack([u, v])
gt_uv = jnp.hstack([gt_u, gt_v])
mse = jnp.mean(jnp.square(uv - gt_uv))
pre_V = jnp.sqrt(jnp.square(u) + jnp.square(v))
gt_V = jnp.sqrt(jnp.square(gt_u) + jnp.square(gt_v))

u_mse = jnp.mean(jnp.square(u - gt_u))
v_mse = jnp.mean(jnp.square(v - gt_v))
V_mse = jnp.mean(jnp.square(pre_V - gt_V))
u_rl2 = jnp.linalg.norm(u - gt_u) / jnp.linalg.norm(gt_u)
v_rl2 = jnp.linalg.norm(v - gt_v) / jnp.linalg.norm(gt_v)
V_rl2 = jnp.linalg.norm(pre_V - gt_V) / jnp.linalg.norm(gt_V)



print ('[Re=%.1f] :  u_MSE = %.2e v_MSE = %.2e V_MSE = %.2e u_rl2 = %.2e v_rl2 = %.2e V_rl2 = %.2e'%(Re, u_mse, v_mse, V_mse, u_rl2, v_rl2, V_rl2))

[Re=5000.0] :  u_MSE = 7.60e-05 v_MSE = 7.00e-05 V_MSE = 1.36e-04 u_rl2 = 4.05e-02 v_rl2 = 4.05e-02 V_rl2 = 3.90e-02
